In [1]:
# 벡터 데이터베이스
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

/tmp/ipykernel_25560/3514318618.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


In [2]:
loader = PyPDFLoader("/workspace/day_02/test.pdf")
pages = loader.load()
print("청크의 수:", len(pages))

청크의 수: 84


In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(pages)
print("분할된 청크의 수:", len(splits))

분할된 청크의 수: 135


In [4]:
# 각 청크의 길이(문자 수)를 저장한 리스트 생성
chunk_lengths = [len(chunk.page_content) for chunk in splits]
max_length = max(chunk_lengths)
min_length = min(chunk_lengths)
avg_length = sum(chunk_lengths) / len(chunk_lengths)

print('청크의 최대 길이:', max_length)
print('청크의 최소 길이:', min_length)
print('청크의 평균 길이:', avg_length)

청크의 최대 길이: 1000
청크의 최소 길이: 1000
청크의 평균 길이: 674.9481481481481


In [5]:
# ChromaDB
import numpy as np
from numpy import dot
from numpy.linalg import norm
import pandas as pd
from langchain_openai import OpenAIEmbeddings

embedding_function = OpenAIEmbeddings( 
    model="text-embedding-qwen3-embedding-8b",
    base_url="http://...:.../v1",
    api_key="lm-studio",
    check_embedding_ctx_length=False, # 토큰 ID 대신 원문 문자열 전송
)

In [6]:
# Docker에 설치한 ChromaDB 사용
import chromadb

chroma_client = chromadb.HttpClient(host="chromadb", port=8000)
print("heartbeat:", chroma_client.heartbeat())  # ns 값 나오면 연결 OK

heartbeat: 1782299569724252470


In [7]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "test.pdf"   # 노트북과 같은 폴더면 파일명만, 아니면 실제 경로로 수정

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print("청크의 수:", len(pages))

청크의 수: 84


In [8]:
import re
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# pages가 다른 셀에서 덮어써졌을 수 있어 항상 새로 로드 (상태 의존 제거)
pages = PyPDFLoader(PDF_PATH).load()
print("원본 페이지:", len(pages))

# 머리글 / 페이지번호 / 차트·표 캡션 soup 제거
HEADER  = re.compile(r'2024\s*KB\s*부동산\s*보고서[^\n]*')
PAGENO  = re.compile(r'^\s*\d{1,3}\s*$', re.M)
CAPTION = re.compile(r'^\s*(그림|표|자료|주)\s*[ⅠⅡⅢIVX\d][^\n]*$', re.M)

def clean_page(text: str) -> str:
    text = HEADER.sub('', text)
    text = CAPTION.sub('', text)
    text = PAGENO.sub('', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

cleaned = []
for p in pages:
    body = clean_page(p.page_content)
    if len(body) < 50:                      # 표·그림만 있던 페이지 스킵
        continue
    cleaned.append(Document(page_content=body, metadata=dict(p.metadata)))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,                          # 한 청크 ≈ 한 논점
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", "。", "! ", "? ", " ", ""],
    keep_separator=True,
)
splits = splitter.split_documents(cleaned)
splits = [d for d in splits if len(d.page_content) >= 80]   # 자투리 제거

lens = [len(d.page_content) for d in splits]
print(f"정리 후 페이지 {len(cleaned)} → 청크 {len(splits)}개 "
      f"| min {min(lens)} / avg {sum(lens)//len(lens)} / max {max(lens)}")

원본 페이지: 84
정리 후 페이지 80 → 청크 215개 | min 83 / avg 374 / max 499


In [9]:
# 청크 길이 통계
chunk_lengths = [len(chunk.page_content) for chunk in splits]
print('청크의 최대 길이:', max(chunk_lengths))
print('청크의 최소 길이:', min(chunk_lengths))
print('청크의 평균 길이:', sum(chunk_lengths) / len(chunk_lengths))

청크의 최대 길이: 499
청크의 최소 길이: 83
청크의 평균 길이: 374.3906976744186


In [10]:
from langchain_chroma import Chroma

COLLECTION = "kb_realestate_2024"

# (선택) 재실행 시 중복 적재 방지 — 기존 컬렉션 삭제
try:
    chroma_client.delete_collection(COLLECTION)
    print("기존 컬렉션 삭제됨")
except Exception as e:
    print("삭제 건너뜀:", e)

삭제 건너뜀: Collection [kb_realestate_2024] does not exist


In [11]:
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding_function,
    client=chroma_client,
    collection_name=COLLECTION,
    collection_metadata={"hnsw:space": "cosine"},  # 비OpenAI 임베딩: cosine 권장
)
print('문서의 수:', vectordb._collection.count())

문서의 수: 215


In [12]:
vectordb = Chroma(
    client=chroma_client,
    collection_name=COLLECTION,
    embedding_function=embedding_function,
)
print('문서의 수:', vectordb._collection.count())

문서의 수: 215


In [13]:
question = "수도권 주택 매매 전망"
top_docs = vectordb.similarity_search(question, k=2)
for i, doc in enumerate(top_docs, 1):
    print(f"문서 {i}:")
    print(f"내용: {doc.page_content[:150]}...")
    print(f"메타데이터: {doc.metadata}")
    print('--' * 20)

문서 1:
내용: 자료: KB경영연구소 
 
지역별로 수도권과 비수도권 모두 하락 전망이 많았으나, 시장 여건 개선에 대한 기대감이 반영되며 
전문가와 공인중개사의 1/3이 수도권 주택 매매가격 상승을 전망하였다. 비수도권에 대해서는 전문가
들이 조금 더 부정적으로 시장을 예측했다. 전...
메타데이터: {'page': 34, 'producer': 'Microsoft® Word 2016', 'author': '손은경', 'title': 'Morning Meeting', 'total_pages': 84, 'page_label': '35', 'source': 'test.pdf', 'moddate': '2024-03-04T15:30:01+09:00', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00'}
----------------------------------------
문서 2:
내용: 자료: KB경영연구소 
 
■ 2024년에도 주택 매매 거래 부진은 지속될 것으로 전망 
2024년 전국 주택 매매 거래량에 대해 전
문가의 71%는 2022~2023년과 유사한 수
준이 될 것으로 전망하였다. 지역별로는 비수
도권에서 주택 매매거래량이 역대 최저였던 ...
메타데이터: {'page_label': '36', 'author': '손은경', 'title': 'Morning Meeting', 'producer': 'Microsoft® Word 2016', 'total_pages': 84, 'page': 35, 'source': 'test.pdf', 'creator': 'Microsoft® Word 2016', 'moddate': '2024-03-04T15:30:01+09:00', 'creationdate': '2024-03-04T15:30:01+09:00'}
----------------------------------------


In [14]:
def instruct_query(task: str, query: str) -> str:
    return f"Instruct: {task}\nQuery: {query}"          # 공식 포맷(Query: 뒤 공백)

def search_rows(task, question, k=5):
    q = instruct_query(task, question) if task else question
    hits = vectordb.similarity_search_with_relevance_scores(q, k=k)
    return [(d.metadata.get("page"), round(s, 3), d.page_content.replace("\n", " "))
            for d, s in hits]

def compare(question, variants, k=3, w=160):
    """같은 질문에 인스트럭션만 바꿔, top 청크가 어떻게 갈리는지 한눈에."""
    print(f"질문: {question!r}\n" + "=" * w)
    seen = {}
    for name, task in variants.items():
        rows = search_rows(task, question, k=k)
        seen[name] = [p for p, _, _ in rows]
        print(f"[{name}] instruct = {task or '(none)'}")
        for p, score, snip in rows:
            print(f"   p{p:>3}  {score:.3f}  {snip[:w]}")
        print("-" * w)
    print("top 페이지 집합:")
    for name, pages in seen.items():
        print(f"   {name:<10} {pages}")
    print("=" * w + "\n")

In [15]:
# 예제 1 — 한 단어, 세 갈래 시멘틱 분리
compare("전세", {
    "가격통계": "Retrieve passages reporting jeonse price changes with concrete "
                "percentages, indices, or numeric figures.",
    "리스크"  : "Retrieve passages about jeonse risks: deposit-return defaults, "
                "reverse-jeonse, and fraud.",
    "정책제도": "Retrieve passages about government measures or institutional "
                "changes affecting the jeonse system.",
}, k=3)

질문: '전세'
[가격통계] instruct = Retrieve passages reporting jeonse price changes with concrete percentages, indices, or numeric figures.
   p 22  0.657  4) 전세 수요 아파트 집중, 입주물량 부족으로 가격 상승 가능성 확대  ■ 2023년 하반기 수도권 아파트를 중심으로 전세가격 회복세를 보였으나 최근 들어 다시 주춤  급격한 금리 인상으로 주택 매매시장뿐 아니라 전세시장도 빠르게 위축되면서 전세가격은 8월 이후  하락세로 전환되었다.
   p  8  0.653  자료: 한국부동산원 자료: 한국부동산원    ■ 전세시장도 하락세가 지속되고 있으나 반등 가능성은 여전히 높은 상황  2023년 주택 전세가격은 전년 대비 5.5% 하락하였다. 매매가격과 마찬가지로 외환위기 직후인  1998년(18.4% 하락) 이후 최대 하락폭이다. 그러나 전세가격은 
   p 37  0.629  전문가와 공인중개사 설문조사 결과, 매매 수요 감소에 따른 전세 수요 증가가 주택 전세가격 상승에  가장 중요한 요인으로 나타났다. 다음으로 신규 입주물량 감소, 전세자금 대출 등 정부 정책 지원에 따 른 수요 증가, 계약 갱신 만료에 따른 이주 수요 증가 순으로 전세가격 상승 요인에 
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[리스크] instruct = Retrieve passages about jeonse risks: deposit-return defaults, reverse-jeonse, and fraud.
   p  8  0.646  자료: 한국부동산원 자료: 한국부동산원    ■ 전세시장도 하락세가 지속되고 있으나 반등 가능성

In [16]:
# 예제 2 — 같은 질문, 이해관계자 관점 전환
compare("지금 집을 사도 될까?", {
    "무주택실수요": "Answer for a first-time homebuyer. Retrieve passages on "
                    "affordability, mortgage burden (PIR, DSR), and end-user demand.",
    "다주택투자": "Answer for a multi-home investor. Retrieve passages on rental "
                  "yield, capital-gain outlook, and promising investment regions.",
    "정책당국"  : "Answer from a regulator's view. Retrieve passages on market-"
                  "stabilization policy and systemic risk.",
}, k=3)

질문: '지금 집을 사도 될까?'
[무주택실수요] instruct = Answer for a first-time homebuyer. Retrieve passages on affordability, mortgage burden (PIR, DSR), and end-user demand.
   p 26  0.630  ■ 고금리에 따른 매수 자금 부담 지속  주택 구매력은 구입하고자 하는 주택가격과 필요한 자금을 조달하는 데 드는 비용에 의해 결정된다.  주택가격이 높으면 그만큼 대출 등을 통해 조달해야 하는 자금이 많아지고 금리가 높으면 원리금 상환 에 필요한 비용 부담이 커진다.  서울 아파트 평
   p 27  0.540  한편 가계 부채 관리 강화로 금리 인하의 효과가 제한적일 수 있다. 정부는 가계 부채의 안정적 관리 를 위해 총부채원리금상환비율(DSR, Debt Service Ratio) 관리 강화, 변동금리 스트레스 DSR 도입2 등 관련 제도  개선을 추진하고 있다. 따라서 금리가 인하되더라도 대
   p 27  0.539  ■ 금리 인하에 따른 매수 심리 회복 가능성이 높지만 시장 영향은 제한적일 전망  최근 매수자의 구매력이 개선되고 있지만 매수 심리 회복으로 이어지지 못하고 있다. 기준금리는  2023년 1월 이후 3.5% 수준을 유지하고 있지만 시중은행 주택담보대출 금리는 하락세를 보였다.  2024
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[다주택투자] instruct = Answer for a multi-home investor. Retrieve passages on rental yield, capital-gain outlook, and promising investment regions.
   p 

In [17]:
# 예제 3 — 극성(polarity) 분리: 하방 vs 상방
compare("2024년 수도권 주택가격 방향", {
    "하방위험": "Retrieve ONLY passages predicting price declines, downside risks, "
                "or a bearish outlook.",
    "상방요인": "Retrieve ONLY passages predicting price recovery, upside drivers, "
                "or a bullish outlook.",
}, k=4)

질문: '2024년 수도권 주택가격 방향'
[하방위험] instruct = Retrieve ONLY passages predicting price declines, downside risks, or a bearish outlook.
   p  2  0.705  Executive Summary 2   주택 매매시장 하락 전망 우세, 부동산 투자 선호도 하락  • 2024년 주택 매매가격 지난해에 이어 올해도 하락 전망 우세, 높은 금리가 가장 큰 부담  부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문 조사 결과, 20
   p 34  0.674  1) 2024년 주택시장 전망  ■ 주택 매매가격, 2024년에도 하락 전망 우세한 가운데 상승 전망 2023년 대비 증가  부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문조사 결과, 2024년 전국 주 택 매매가격은 하락세가 이어질 것이라는 전망이 우세하였다.
   p  1  0.639  Executive Summary 1   2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목  주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정  가능성이 큰 상황이다. 매수 수요 위축으로 주택 매매 거래량이 급감
   p 36  0.636  ■ 주택 전세가격, 비수도권 하락 전망이 우세한 가운데 수도권 전망은 엇갈려  2024년 전국 주택 전세가격에 대해 전문가의 53%, 공인중개사의 61%가 하락을 전망하였다. 하락 폭에 대해서는 3% 이하가 될 것이라는 의견이 많았다. 2023년 설문조사 결과와 비교해 하락 전망은  크
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[

In [18]:
# 예제 4 — 사실 유형 분리: 숫자 vs 근거
compare("공인중개사의 2024년 집값 전망", {
    "응답수치": "Retrieve the exact survey response percentages — the share answering "
                "rise, fall, or flat.",
    "전망근거": "Retrieve the reasons or rationale respondents gave, not the numbers.",
}, k=3)

질문: '공인중개사의 2024년 집값 전망'
[응답수치] instruct = Retrieve the exact survey response percentages — the share answering rise, fall, or flat.
   p 34  0.765  1) 2024년 주택시장 전망  ■ 주택 매매가격, 2024년에도 하락 전망 우세한 가운데 상승 전망 2023년 대비 증가  부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문조사 결과, 2024년 전국 주 택 매매가격은 하락세가 이어질 것이라는 전망이 우세하였다.
   p  2  0.706  Executive Summary 2   주택 매매시장 하락 전망 우세, 부동산 투자 선호도 하락  • 2024년 주택 매매가격 지난해에 이어 올해도 하락 전망 우세, 높은 금리가 가장 큰 부담  부동산시장 전문가와 공인중개사, 자산관리전문가(PB)를 대상으로 한 설문 조사 결과, 20
   p 36  0.664  ■ 주택 전세가격, 비수도권 하락 전망이 우세한 가운데 수도권 전망은 엇갈려  2024년 전국 주택 전세가격에 대해 전문가의 53%, 공인중개사의 61%가 하락을 전망하였다. 하락 폭에 대해서는 3% 이하가 될 것이라는 의견이 많았다. 2023년 설문조사 결과와 비교해 하락 전망은  크
----------------------------------------------------------------------------------------------------------------------------------------------------------------
[전망근거] instruct = Retrieve the reasons or rationale respondents gave, not the numbers.
   p 34  0.730  1) 2024년 주택시장 전망  ■ 주택 매매가격, 2024년에도 하락 전망 우세한 가운데 상승 전망 2023년 대비 증가  부동